# Continuity Widget: Exploring the $\varepsilon$-$\delta$ Definition

Math 140H, Honors Calculus I

## What this notebook does

Recall the $\varepsilon$-$\delta$ definition: $f$ is continuous at $x_0$ if for every $\varepsilon > 0$
there is a $\delta > 0$ so that
$$
|x - x_0| < \delta \ \Longrightarrow\ |f(x) - f(x_0)| < \varepsilon.
$$
The point of the definition is a *game*: however small a target window $\varepsilon$ around $f(x_0)$
you are handed, you can always find a $\delta$-neighborhood of $x_0$ whose whole image under $f$ fits
inside that window.

The widget below runs this game in reverse, which is easier to see: you pick $x_0$ and a $\delta$, and
it shows you the *smallest* $\varepsilon$-window the $\delta$-neighborhood actually lands in, namely the
true range of $f$-values over $(x_0-\delta, x_0+\delta)$. Watch what happens to that window as you drag
$\delta \to 0$:

- At a point where $f$ is continuous, the window's width shrinks to $0$.
- At a jump, the window's width stops shrinking once $\delta$ is small enough that both sides of the
  jump are still inside the neighborhood — no matter how much smaller you make $\delta$ after that.

**How to use it:** click any point on the blue curve to choose $x_0$; the $\delta$-slider below the
plot then becomes active. Dragging it slides the boundary of the $\delta$-neighborhood, and you can
watch the orange "tunnel" (the region on the graph carved out by that neighborhood together with its
image) and the red bar on the $y$-axis (the resulting $\varepsilon$-window) shrink or grow to match.

In [1]:
# numpy      -- holds the sampled function as plain arrays (x, y = f(x)); the widget never calls
#               f itself, only looks up values you already computed. That means it works just as
#               well for a formula (np.sin(x)) as for data you only know at finitely many points.
# plotly.graph_objects -- builds the interactive figure (as a FigureWidget, which can be updated
#               in place after you click on it, instead of redrawing a brand new picture).
# ipywidgets -- provides the delta-slider and the little text readout underneath the plot, and
#               glues them together with the figure.
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets

## The widget

`continuity_widget(x, y, func_name)` takes a sampled function -- a numpy array `x` of $x$-values and
a numpy array `y` of the corresponding $f(x)$-values -- and returns the interactive display.

The key computation happens in `refresh`: given the clicked point $x_0$ and the current slider value
$\delta$, it looks at every sample point whose $x$ lies in $(x_0-\delta, x_0+\delta)$ (a boolean mask,
exactly the discrete stand-in for "the $\delta$-neighborhood of $x_0$") and takes the min and max of
$f$ over just those points. That `[y_lo, y_hi]` interval *is* the tightest $\varepsilon$-window this
$\delta$ actually buys you -- the true image of the neighborhood, not a guess. (The textbook
$\varepsilon$ is then whichever of $f(x_0)-y_{\text{lo}}$, $y_{\text{hi}}-f(x_0)$ is larger, since the
definition wants a window centered at $f(x_0)$; the status line below the plot reports that number
too.)

In [2]:
def continuity_widget(x, y, func_name="f", title=None):
    """
    Interactive epsilon-delta explorer for a function sampled as arrays x, f(x) = y.

    Click a point on the curve to pick x0; a slider for delta (the half-width of the
    x-neighborhood) then becomes active. As delta changes, the widget shades the region
    of the graph swept out by the delta-neighborhood together with its true range of
    f-values (the "tunnel"), and marks that same range as an interval on the y-axis.

    Parameters
    ----------
    x, y : 1-D numpy arrays of the same length
        The sampled function: y[i] = f(x[i]). Use a fine, evenly spaced x (e.g. from
        np.linspace) so the discrete neighborhood mask below is a good stand-in for a
        real interval.
    func_name : str
        Label used for the function in the axis title and status line, e.g. "f".
    title : str, optional
        Plot title; a sensible default is generated if omitted.

    Returns
    -------
    ipywidgets.VBox
        Display this (as the last line of a cell, or via IPython.display.display) to
        show the figure, the delta-slider, and the status readout stacked vertically.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if x.shape != y.shape:
        raise ValueError("x and y must have the same shape")
    # Sort by x so "the neighborhood of x0" is a contiguous run of the array, and so the
    # line trace does not zig-zag if the caller passed x out of order.
    order = np.argsort(x)
    x, y = x[order], y[order]

    # Fixed axis ranges, computed once from the data. Keeping these fixed (rather than
    # letting plotly autoscale on every update) is what makes the y-axis indicator bar
    # -- drawn at the *left edge* of the plot -- land in the same place every time.
    x_span = x.max() - x.min()
    pad = 0.06 * x_span
    x_left, x_right = x.min() - pad, x.max() + pad
    y_span = y.max() - y.min()
    y_pad = 0.10 * y_span if y_span > 0 else 1.0
    y_bottom, y_top = y.min() - y_pad, y.max() + y_pad

    # The function's graph. mode="lines+markers" with small, faint markers keeps the
    # curve looking smooth while still giving plotly something at every sample point to
    # register a click on -- clicking on a bare line (no markers) is unreliable.
    curve = go.Scatter(
        x=x, y=y, mode="lines+markers",
        line=dict(color="#1f77b4", width=2),
        marker=dict(size=4, color="#1f77b4", opacity=0.35),
        name=func_name, hoverinfo="x+y",
    )
    # A second, initially empty trace just to highlight whichever point the student
    # clicked; we overwrite its single (x, y) pair every time a new point is picked.
    selected_point = go.Scatter(
        x=[], y=[], mode="markers",
        marker=dict(size=11, color="#d62728", line=dict(width=1, color="white")),
        name="x0", hoverinfo="skip", showlegend=False,
    )

    fig = go.FigureWidget(data=[curve, selected_point])
    fig.update_layout(
        template="plotly_white",
        title=title or f"Continuity at a point: click the graph of {func_name}",
        xaxis=dict(title="x", range=[x_left, x_right], zeroline=False),
        yaxis=dict(title=f"{func_name}(x)", range=[y_bottom, y_top], zeroline=False),
        margin=dict(t=60, b=40, l=60, r=20),
        width=700, height=460,
        showlegend=False,
    )

    # delta ranges from "a few sample spacings" up to a quarter of the whole domain --
    # small enough to demonstrate delta -> 0, large enough to matter visually.
    delta_max = x_span / 2
    delta_min = max(x_span / 500, 1e-6)
    slider = widgets.FloatSlider(
        value=delta_max / 4, min=delta_min, max=delta_max, step=delta_min,
        description="delta", disabled=True, continuous_update=True,
        readout_format=".4f", layout=widgets.Layout(width="620px"),
    )
    status = widgets.HTML(value="<i>Click a point on the curve to pick x&#8320;.</i>")

    # Remembers which sample index is currently selected (None until the first click).
    state = {"idx": None}

    def neighborhood_shapes(x0, y0, delta):
        """Everything the plot needs to draw for the current (x0, delta): the tunnel
        rectangle, its delta-boundary and epsilon-boundary guide lines, and the bold
        y-axis indicator bar. Returns the shape list plus the y-range it highlights."""
        mask = np.abs(x - x0) <= delta
        if not mask.any():
            # delta smaller than the sample spacing: fall back to the single nearest
            # point, so the display still degrades gracefully instead of erroring.
            mask = np.array([np.argmin(np.abs(x - x0))])
        y_lo, y_hi = y[mask].min(), y[mask].max()
        d_lo, d_hi = x0 - delta, x0 + delta
        shapes = [
            # The "tunnel": the box the graph is confined to over this delta-neighborhood.
            dict(type="rect", xref="x", yref="y", x0=d_lo, x1=d_hi, y0=y_lo, y1=y_hi,
                 fillcolor="rgba(255,140,0,0.25)",
                 line=dict(color="rgba(230,120,0,0.8)", width=1), layer="below"),
            # Vertical dotted lines marking x0 - delta and x0 + delta.
            dict(type="line", xref="x", yref="paper", x0=d_lo, x1=d_lo, y0=0, y1=1,
                 line=dict(color="rgba(230,120,0,0.8)", width=1, dash="dot")),
            dict(type="line", xref="x", yref="paper", x0=d_hi, x1=d_hi, y0=0, y1=1,
                 line=dict(color="rgba(230,120,0,0.8)", width=1, dash="dot")),
            # Horizontal dotted guide lines connecting the tunnel to the y-axis bar.
            dict(type="line", xref="x", yref="y", x0=x_left, x1=d_hi, y0=y_lo, y1=y_lo,
                 line=dict(color="gray", width=1, dash="dot")),
            dict(type="line", xref="x", yref="y", x0=x_left, x1=d_hi, y0=y_hi, y1=y_hi,
                 line=dict(color="gray", width=1, dash="dot")),
            # The y-axis indicator: a bold bar sitting right on the axis, spanning
            # exactly the range of f-values the delta-neighborhood produces.
            dict(type="line", xref="x", yref="y", x0=x_left, x1=x_left, y0=y_lo, y1=y_hi,
                 line=dict(color="#d62728", width=9)),
        ]
        return shapes, y_lo, y_hi

    def refresh(x0, y0, delta):
        shapes, y_lo, y_hi = neighborhood_shapes(x0, y0, delta)
        with fig.batch_update():
            fig.layout.shapes = shapes
            fig.data[1].x, fig.data[1].y = [x0], [y0]
        eps = max(y0 - y_lo, y_hi - y0)  # textbook epsilon: window centered at f(x0)
        status.value = (
            f"x&#8320; = {x0:.4g},  {func_name}(x&#8320;) = {y0:.4g} &nbsp;|&nbsp; "
            f"delta = {delta:.4g} &nbsp;&rarr;&nbsp; "
            f"{func_name} stays in [{y_lo:.4g}, {y_hi:.4g}] on (x&#8320;-delta, x&#8320;+delta) "
            f"&nbsp;|&nbsp; epsilon = {eps:.4g}"
        )

    def on_click(trace, points, click_state):
        if not points.point_inds:
            return
        idx = points.point_inds[0]
        state["idx"] = idx
        slider.disabled = False
        refresh(x[idx], y[idx], slider.value)

    fig.data[0].on_click(on_click)

    def on_slider_change(change):
        idx = state["idx"]
        if idx is None:
            return
        refresh(x[idx], y[idx], change["new"])

    slider.observe(on_slider_change, names="value")

    return widgets.VBox([fig, slider, status])

## Try it: a continuous function

$f(x) = x^3 - 3x$ on $[-3, 3]$. Click a few different points on the curve, then narrow $\delta$ with
the slider.

PREDICT (before you drag the slider all the way down): at a point where the curve is fairly flat, do
you expect the same $\delta$ to produce a *wider* or *narrower* $\varepsilon$-window than at a point
where the curve is steep? Check your prediction at $x_0 \approx 0$ (flat, an inflection point) versus
$x_0 \approx 2$ (steep).

In [3]:
x_vals = np.linspace(-3, 3, 600)
y_vals = x_vals**3 - 3 * x_vals

continuity_widget(x_vals, y_vals, func_name="f")

    'data': [{'hoverinfo': 'x+y',
              'line': {'color': '#1f77b4', 'wi…

## Contrast: a function with a jump discontinuity

$$
g(x) = \begin{cases} -1 + 0.3x & x < 0 \\ \ \ 1 + 0.3x & x \ge 0 \end{cases}
$$

Click right at $x_0 = 0$ and narrow $\delta$ as far as it goes.

VERIFY: for a continuous function, the epsilon reading in the status line keeps shrinking as
$\delta \to 0$. At $x_0 = 0$ here, it should stop shrinking once $\delta$ is small enough that the
tunnel no longer reaches across the jump -- after that, no smaller $\delta$ helps, which is exactly
why $g$ fails the $\varepsilon$-$\delta$ definition at that point. Compare that to clicking a point
away from the jump, e.g. $x_0 = 1.5$, where narrowing $\delta$ keeps shrinking the window as usual.

In [4]:
x_vals2 = np.linspace(-3, 3, 601)
y_vals2 = np.where(x_vals2 < 0, -1.0, 1.0) + 0.3 * x_vals2

continuity_widget(x_vals2, y_vals2, func_name="g")

    'data': [{'hoverinfo': 'x+y',
              'line': {'color': '#1f77b4', 'wi…

## Try it yourself

Replace the definition of `y_vals3` below with a function of your own choosing (built out of numpy,
e.g. `np.sin`, `np.abs`, `1 / x_vals3` with a domain avoiding $0$, a piecewise function via
`np.where` like `g` above, ...). Click near a point you suspect is continuous, and near one you
suspect is not, and see whether the widget agrees with your intuition.

In [ ]:
x_vals3 = np.linspace(-3, 3, 600)
y_vals3 = np.sin(x_vals3)  # replace with a function of your own

continuity_widget(x_vals3, y_vals3, func_name="h")